In [1]:
!pip install lightgbm xgboost --upgrade


   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ------- -------------------------------- 0.3/1.5 MB ? eta -:--:--
   --------------------- ------------------ 0.8/1.5 MB 4.2 MB/s eta 0:00:01
   ------------------------------------ --- 1.3/1.5 MB 2.8 MB/s eta 0:00:01
   ------------------------------------ --- 1.3/1.5 MB 2.8 MB/s eta 0:00:01
   ------------------------------------ --- 1.3/1.5 MB 2.8 MB/s eta 0:00:01
   ---------------------------------------- 1.5/1.5 MB 1.3 MB/s eta 0:00:00
   ---------------------------------------- 0.0/150.0 MB ? eta -:--:--
   ---------------------------------------- 0.8/150.0 MB 4.8 MB/s eta 0:00:32
   ---------------------------------------- 1.8/150.0 MB 4.6 MB/s eta 0:00:33
    --------------------------------------- 3.4/150.0 MB 5.6 MB/s eta 0:00:27
   - -------------------------------------- 5.0/150.0 MB 6.3 MB/s eta 0:00:24
   - -------------------------------------- 7.3/150.0 MB 7.1 MB/s eta 0:00:21
   -- ---------------

In [3]:
import pandas as pd
import numpy as np
from datetime import datetime
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import StackingClassifier

# 1. Load the dataset
df = pd.read_csv('bot_detection_data.csv')

# 2. Feature engineering: compute account age in days
df['Created At'] = pd.to_datetime(df['Created At'])
df['account_age_days'] = (datetime.now() - df['Created At']).dt.days

# 3. Prepare features and target
features = ['Retweet Count', 'Mention Count', 'Follower Count', 'Verified', 'account_age_days']
X = df[features].copy()
X['Verified'] = X['Verified'].astype(int)
y = df['Bot Label']

# 4. Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 5. Build preprocessing pipeline
numeric_features = ['Retweet Count', 'Mention Count', 'Follower Count', 'account_age_days']
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])
preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', 'passthrough', ['Verified'])
])

# 6. Define individual model pipelines
xgb_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42))
])
lgbm_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LGBMClassifier(random_state=42))
])

# 7. Train & evaluate XGBoost and LightGBM
for name, pipe in [('XGBoost', xgb_pipeline), ('LightGBM', lgbm_pipeline)]:
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    print(f"\n=== {name} Classification Report ===")
    print(classification_report(y_test, y_pred))
    print(f"=== {name} Confusion Matrix ===")
    print(confusion_matrix(y_test, y_pred))

# 8. Build stacking ensemble of XGBoost + LightGBM
stacking_clf = StackingClassifier(
    estimators=[
        ('xgb', XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)),
        ('lgbm', LGBMClassifier(random_state=42))
    ],
    final_estimator=LogisticRegression(),
    cv=5,
    stack_method='predict_proba'
)
stack_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('stacking', stacking_clf)
])

# 9. Train & evaluate stacking ensemble
stack_pipeline.fit(X_train, y_train)
y_pred_stack = stack_pipeline.predict(X_test)
print("\n=== Stacking Ensemble Classification Report ===")
print(classification_report(y_test, y_pred_stack))
print("=== Stacking Ensemble Confusion Matrix ===")
print(confusion_matrix(y_test, y_pred_stack))

C:\Users\Admin\anaconda3\Lib\site-packages\xgboost\training.py:183: UserWarning: [11:58:31] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



=== XGBoost Classification Report ===
              precision    recall  f1-score   support

           0       0.50      0.50      0.50      4996
           1       0.50      0.50      0.50      5004

    accuracy                           0.50     10000
   macro avg       0.50      0.50      0.50     10000
weighted avg       0.50      0.50      0.50     10000

=== XGBoost Confusion Matrix ===
[[2479 2517]
 [2505 2499]]
[LightGBM] [Info] Number of positive: 20014, number of negative: 19986
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000222 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 621
[LightGBM] [Info] Number of data points in the train set: 40000, number of used features: 5
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500350 -> initscore=0.001400
[LightGBM] [Info] Start training from score 0.001400

=== LightGBM Classi

C:\Users\Admin\anaconda3\Lib\site-packages\xgboost\training.py:183: UserWarning: [11:58:31] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
C:\Users\Admin\anaconda3\Lib\site-packages\xgboost\training.py:183: UserWarning: [11:58:31] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


[LightGBM] [Info] Number of positive: 20014, number of negative: 19986
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000170 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 621
[LightGBM] [Info] Number of data points in the train set: 40000, number of used features: 5
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500350 -> initscore=0.001400
[LightGBM] [Info] Start training from score 0.001400


C:\Users\Admin\anaconda3\Lib\site-packages\xgboost\training.py:183: UserWarning: [11:58:32] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


[LightGBM] [Info] Number of positive: 16011, number of negative: 15989
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000180 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 621
[LightGBM] [Info] Number of data points in the train set: 32000, number of used features: 5
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500344 -> initscore=0.001375
[LightGBM] [Info] Start training from score 0.001375
[LightGBM] [Info] Number of positive: 16011, number of negative: 15989
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000167 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 621
[LightGBM] [Info] Number of data points in the train set: 32000, number of used features: 5
[LightGBM] [Info] [binar

In [11]:

import pandas as pd
import numpy as np
from datetime import datetime
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, confusion_matrix
from xgboost import XGBClassifier

# 1. Load the dataset
df = pd.read_csv('bot_detection_data.csv')

# 2. Feature engineering: compute account age in days
df['Created At'] = pd.to_datetime(df['Created At'])
df['account_age_days'] = (datetime.now() - df['Created At']).dt.days

# 3. Prepare features and target
numeric_features = ['Retweet Count', 'Mention Count', 'Follower Count', 'account_age_days', 'Verified']
text_feature = 'Tweet'
target = 'Bot Label'

# Ensure 'Verified' is int
df['Verified'] = df['Verified'].astype(int)

X = df[numeric_features + [text_feature]]
y = df[target]

# 4. Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 5. Build preprocessing pipeline
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('text', TfidfVectorizer(max_features=5000, ngram_range=(1,2)), text_feature)
])

# 6. Build full pipeline with XGBoost classifier
pipeline = Pipeline(steps=[
    ('pre', preprocessor),
    ('clf', XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42))
])

# 7. Define hyperparameter search space
param_dist = {
    'clf__n_estimators': [100, 200, 300, 500],
    'clf__max_depth': [3, 5, 7, 10],
    'clf__learning_rate': [0.01, 0.05, 0.1, 0.2],
    'clf__subsample': [0.6, 0.8, 1.0],
    'clf__colsample_bytree': [0.6, 0.8, 1.0]
}

# 8. Setup RandomizedSearchCV
search = RandomizedSearchCV(
    pipeline,
    param_distributions=param_dist,
    n_iter=20,
    cv=3,
    scoring='f1',
    verbose=2,
    n_jobs=-1,
    random_state=42
)

# 9. Run hyperparameter search
search.fit(X_train, y_train)

# 10. Evaluate best model on test set
best_model = search.best_estimator_
y_pred = best_model.predict(X_test)

print("Best Parameters:\n", search.best_params_)
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

Fitting 3 folds for each of 20 candidates, totalling 60 fits


C:\Users\Admin\anaconda3\Lib\site-packages\xgboost\training.py:183: UserWarning: [12:09:22] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Best Parameters:
 {'clf__subsample': 1.0, 'clf__n_estimators': 200, 'clf__max_depth': 7, 'clf__learning_rate': 0.01, 'clf__colsample_bytree': 0.6}

Classification Report:
               precision    recall  f1-score   support

           0       0.49      0.28      0.36      4996
           1       0.50      0.72      0.59      5004

    accuracy                           0.50     10000
   macro avg       0.50      0.50      0.47     10000
weighted avg       0.50      0.50      0.47     10000

Confusion Matrix:
 [[1387 3609]
 [1421 3583]]


In [17]:
!pip install catboost

  Using cached graphviz-0.20.3-py3-none-any.whl.metadata (12 kB)
   ---------------------------------------- 0.0/102.4 MB ? eta -:--:--
   ---------------------------------------- 0.5/102.4 MB 4.2 MB/s eta 0:00:25
    --------------------------------------- 2.1/102.4 MB 6.2 MB/s eta 0:00:17
   - -------------------------------------- 3.4/102.4 MB 6.3 MB/s eta 0:00:16
   - -------------------------------------- 3.9/102.4 MB 5.0 MB/s eta 0:00:20
   -- ------------------------------------- 5.2/102.4 MB 5.6 MB/s eta 0:00:18
   -- ------------------------------------- 6.3/102.4 MB 5.6 MB/s eta 0:00:18
   --- ------------------------------------ 8.1/102.4 MB 5.9 MB/s eta 0:00:17
   --- ------------------------------------ 8.7/102.4 MB 5.8 MB/s eta 0:00:17
   --- ------------------------------------ 9.4/102.4 MB 5.3 MB/s eta 0:00:18
   ---- ----------------------------------- 11.3/102.4 MB 5.6 MB/s eta 0:00:17
   ---- ----------------------------------- 12.1/102.4 MB 5.6 MB/s eta 0:00:17
   -

In [21]:
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

imb_pipeline = ImbPipeline([
    ("pre", preprocessor),         # your ColumnTransformer (TFIDF + scaler)
    ("smote", SMOTE(random_state=42)),
    ("clf", XGBClassifier(
        n_estimators=200,
        max_depth=7,
        learning_rate=0.01,
        subsample=1.0,
        colsample_bytree=0.6,
        eval_metric="logloss",
        use_label_encoder=False,
        random_state=42
    ))
])

imb_pipeline.fit(X_train, y_train)
y_pred = imb_pipeline.predict(X_test)
print(classification_report(y_test, y_pred))


C:\Users\Admin\anaconda3\Lib\site-packages\xgboost\training.py:183: UserWarning: [12:14:33] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


              precision    recall  f1-score   support

           0       0.50      0.28      0.36      4996
           1       0.50      0.72      0.59      5004

    accuracy                           0.50     10000
   macro avg       0.50      0.50      0.47     10000
weighted avg       0.50      0.50      0.47     10000

